In [5]:
# ============================================================
# V4 — SINGLE-CELL FEEDBACK-DRIVEN LLM ROUTER
# ============================================================
#
# V4 flow:
#
# Evaluation data
#      ↓
# V3 routing policy
#      ↓
# Simulated model outcome
#      ↓
# quality / latency / cost / success
#      ↓
# Observed performance
#      ↓
# V4 feedback-driven routing
#      ↓
# Save local artifacts
#      ↓
# Optional S3 upload
#
# NOTE:
# Model outcomes are simulated in this V4 experiment.
# Later, these can be replaced with real model inference.
# ============================================================

import os
import json
import boto3
import numpy as np
import pandas as pd


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = "/home/sagemaker-user/adaptive-llm-routing"

V4_DIR = os.path.join(
    PROJECT_ROOT,
    "notebooks",
    "data",
    "evaluation",
    "v4"
)

os.makedirs(
    V4_DIR,
    exist_ok=True
)


print("=" * 100)
print("V4 — FEEDBACK-DRIVEN LLM ROUTER")
print("=" * 100)


# ============================================================
# 2. FIND EVALUATION DATASET AUTOMATICALLY
# ============================================================

candidate_files = []

for root, dirs, files in os.walk(PROJECT_ROOT):

    dirs[:] = [
        d for d in dirs
        if d not in {
            ".git",
            "__pycache__",
            ".venv",
            "v4"
        }
    ]

    for filename in files:

        if filename.endswith(
            (".csv", ".json")
        ):

            candidate_files.append(
                os.path.join(
                    root,
                    filename
                )
            )


def normalize_complexity(value):

    value = (
        str(value)
        .lower()
        .strip()
    )

    mapping = {

        "0": "simple",
        "1": "medium",
        "2": "complex",

        "simple": "simple",
        "medium": "medium",
        "complex": "complex"
    }

    return mapping.get(
        value,
        value
    )


evaluation_df = None
evaluation_path = None


for path in candidate_files:

    if "/v4/" in path:
        continue

    try:

        if path.endswith(".csv"):

            temp = pd.read_csv(
                path
            )

        else:

            with open(
                path,
                "r"
            ) as f:

                raw = json.load(f)


            if isinstance(
                raw,
                list
            ):

                temp = pd.DataFrame(
                    raw
                )

            else:

                continue


        if len(temp) < 500:
            continue


        columns = {
            str(c).lower(): c
            for c in temp.columns
        }


        # Find query column
        query_col = None

        for name in [
            "query",
            "question",
            "prompt",
            "instruction",
            "text"
        ]:

            if name in columns:

                query_col = columns[name]
                break


        # Find complexity column
        complexity_col = None

        for name in [
            "complexity",
            "label",
            "complexity_label",
            "category",
            "class"
        ]:

            if name in columns:

                complexity_col = columns[name]
                break


        if (
            query_col is None
            or
            complexity_col is None
        ):

            continue


        temp = temp.copy()


        temp["query"] = (
            temp[query_col]
            .astype(str)
            .str.strip()
        )


        temp["complexity"] = (
            temp[complexity_col]
            .apply(
                normalize_complexity
            )
        )


        valid = {
            "simple",
            "medium",
            "complex"
        }


        temp = temp[
            temp["complexity"].isin(
                valid
            )
        ]


        temp = temp[
            temp["query"].str.len() > 0
        ]


        if len(temp) >= 500:

            evaluation_df = (
                temp[
                    [
                        "query",
                        "complexity"
                    ]
                ]
                .reset_index(
                    drop=True
                )
            )

            evaluation_path = path

            break


    except Exception:
        continue


if evaluation_df is None:

    raise FileNotFoundError(
        "Could not find a suitable "
        "evaluation dataset with "
        "query + complexity columns."
    )


print(
    "\nEvaluation dataset:"
)

print(
    evaluation_path
)

print(
    "\nTotal evaluation queries:",
    len(evaluation_df)
)

print(
    "\nComplexity distribution:"
)

print(
    evaluation_df[
        "complexity"
    ].value_counts()
)


# ============================================================
# 3. V3 MODEL PROFILES
# ============================================================

v3_model_profiles = {

    "small_model": {

        "quality": 0.82,

        "latency_ms": 200,

        "cost": 0.10
    },

    "medium_model": {

        "quality": 0.91,

        "latency_ms": 500,

        "cost": 0.30
    },

    "large_model": {

        "quality": 0.97,

        "latency_ms": 1200,

        "cost": 1.00
    }
}


# ============================================================
# 4. QUALITY REQUIREMENTS
# ============================================================

quality_requirement = {

    "simple": 0.80,

    "medium": 0.90,

    "complex": 0.95
}


COST_WEIGHT = 0.55
LATENCY_WEIGHT = 0.45


# ============================================================
# 5. V3 ROUTING POLICY
# ============================================================

def v3_route(complexity):

    required_quality = (
        quality_requirement[
            complexity
        ]
    )


    eligible = {}

    for model_name, profile in (
        v3_model_profiles.items()
    ):

        if (
            profile["quality"]
            >=
            required_quality
        ):

            eligible[
                model_name
            ] = profile


    if not eligible:

        return max(

            v3_model_profiles,

            key=lambda name:
            v3_model_profiles[
                name
            ]["quality"]
        )


    costs = [
        p["cost"]
        for p in eligible.values()
    ]


    latencies = [
        p["latency_ms"]
        for p in eligible.values()
    ]


    min_cost = min(costs)
    max_cost = max(costs)

    min_latency = min(latencies)
    max_latency = max(latencies)


    scores = {}


    for model_name, profile in (
        eligible.items()
    ):

        if max_cost == min_cost:

            normalized_cost = 0.0

        else:

            normalized_cost = (

                profile["cost"]
                -
                min_cost

            ) / (

                max_cost
                -
                min_cost
            )


        if max_latency == min_latency:

            normalized_latency = 0.0

        else:

            normalized_latency = (

                profile["latency_ms"]
                -
                min_latency

            ) / (

                max_latency
                -
                min_latency
            )


        scores[
            model_name
        ] = (

            COST_WEIGHT
            *
            normalized_cost

            +

            LATENCY_WEIGHT
            *
            normalized_latency
        )


    return min(
        scores,
        key=scores.get
    )


# ============================================================
# 6. SIMULATED ACTUAL MODEL PERFORMANCE
# ============================================================
#
# These are NOT used directly by V4 routing.
#
# They represent the "real outcomes" that V4 supposedly
# observes after models answer requests.
#
# In a future Bedrock version, these values will be replaced
# by actual measurements.
# ============================================================

simulated_performance = {

    "simple": {

        "small_model": {
            "quality": 0.91,
            "latency_ms": 190,
            "cost": 0.09
        },

        "medium_model": {
            "quality": 0.94,
            "latency_ms": 480,
            "cost": 0.29
        },

        "large_model": {
            "quality": 0.98,
            "latency_ms": 1150,
            "cost": 0.98
        }
    },

    "medium": {

        "small_model": {
            "quality": 0.76,
            "latency_ms": 210,
            "cost": 0.10
        },

        "medium_model": {
            "quality": 0.90,
            "latency_ms": 510,
            "cost": 0.31
        },

        "large_model": {
            "quality": 0.97,
            "latency_ms": 1190,
            "cost": 1.00
        }
    },

    "complex": {

        "small_model": {
            "quality": 0.55,
            "latency_ms": 220,
            "cost": 0.11
        },

        "medium_model": {
            "quality": 0.80,
            "latency_ms": 530,
            "cost": 0.32
        },

        "large_model": {
            "quality": 0.96,
            "latency_ms": 1220,
            "cost": 1.03
        }
    }
}


# ============================================================
# 7. GENERATE FEEDBACK
# ============================================================

rng = np.random.default_rng(
    42
)

feedback = []


for _, row in evaluation_df.iterrows():

    query = row["query"]

    complexity = row[
        "complexity"
    ]


    # -----------------------------------------------
    # V3 selects a model
    # -----------------------------------------------

    selected_model = v3_route(
        complexity
    )


    # -----------------------------------------------
    # Simulated real outcome
    # -----------------------------------------------

    base = (
        simulated_performance[
            complexity
        ][
            selected_model
        ]
    )


    quality = rng.normal(
        loc=base["quality"],
        scale=0.04
    )

    quality = float(
        np.clip(
            quality,
            0.0,
            1.0
        )
    )


    latency = rng.normal(
        loc=base["latency_ms"],
        scale=(
            base["latency_ms"]
            *
            0.08
        )
    )

    latency = float(
        max(
            1,
            latency
        )
    )


    cost = rng.normal(
        loc=base["cost"],
        scale=(
            base["cost"]
            *
            0.05
        )
    )

    cost = float(
        max(
            0.001,
            cost
        )
    )


    success_probability = (
        0.95
        +
        0.05 * quality
    )


    success = int(
        rng.random()
        <
        success_probability
    )


    feedback.append({

        "query":
            query,

        "complexity":
            complexity,

        "selected_model":
            selected_model,

        "quality":
            round(
                quality,
                4
            ),

        "latency_ms":
            round(
                latency,
                2
            ),

        "cost":
            round(
                cost,
                4
            ),

        "success":
            success
    })


feedback_df = pd.DataFrame(
    feedback
)


# ============================================================
# 8. SAVE RAW FEEDBACK
# ============================================================

feedback_path = os.path.join(
    V4_DIR,
    "routing_feedback.csv"
)


feedback_df.to_csv(
    feedback_path,
    index=False
)


# ============================================================
# 9. LEARN OBSERVED PERFORMANCE
# ============================================================

performance_df = (

    feedback_df

    .groupby(
        [
            "complexity",
            "selected_model"
        ]
    )

    .agg(

        queries=(
            "query",
            "count"
        ),

        avg_quality=(
            "quality",
            "mean"
        ),

        avg_latency_ms=(
            "latency_ms",
            "mean"
        ),

        avg_cost=(
            "cost",
            "mean"
        ),

        success_rate=(
            "success",
            "mean"
        )
    )

    .reset_index()
)


performance_df[
    "avg_quality"
] = performance_df[
    "avg_quality"
].round(4)


performance_df[
    "avg_latency_ms"
] = performance_df[
    "avg_latency_ms"
].round(2)


performance_df[
    "avg_cost"
] = performance_df[
    "avg_cost"
].round(4)


performance_df[
    "success_rate"
] = performance_df[
    "success_rate"
].round(4)


# ============================================================
# 10. SAVE PERFORMANCE CSV
# ============================================================

performance_csv = os.path.join(
    V4_DIR,
    "model_performance.csv"
)


performance_df.to_csv(
    performance_csv,
    index=False
)


# ============================================================
# 11. SAVE PERFORMANCE JSON
# ============================================================

observed_performance = {}


for _, row in performance_df.iterrows():

    complexity = (
        row["complexity"]
    )

    model_name = (
        row["selected_model"]
    )


    if complexity not in (
        observed_performance
    ):

        observed_performance[
            complexity
        ] = {}


    observed_performance[
        complexity
    ][model_name] = {

        "queries":
            int(
                row["queries"]
            ),

        "quality":
            float(
                row["avg_quality"]
            ),

        "latency_ms":
            float(
                row["avg_latency_ms"]
            ),

        "cost":
            float(
                row["avg_cost"]
            ),

        "success_rate":
            float(
                row["success_rate"]
            )
    }


performance_json = os.path.join(
    V4_DIR,
    "model_performance.json"
)


with open(
    performance_json,
    "w"
) as f:

    json.dump(
        observed_performance,
        f,
        indent=2
    )


# ============================================================
# 12. BUILD V4 FEEDBACK-DRIVEN ROUTER
# ============================================================

SUCCESS_THRESHOLD = 0.95


def route_v4(complexity):

    required_quality = (
        quality_requirement[
            complexity
        ]
    )


    available = (
        observed_performance.get(
            complexity,
            {}
        )
    )


    # --------------------------------------------------------
    # Only models that satisfy quality + reliability
    # --------------------------------------------------------

    eligible = {}


    for model_name, profile in (
        available.items()
    ):

        if (

            profile["quality"]
            >=
            required_quality

            and

            profile["success_rate"]
            >=
            SUCCESS_THRESHOLD
        ):

            eligible[
                model_name
            ] = profile


    # --------------------------------------------------------
    # Fallback to highest observed quality
    # --------------------------------------------------------

    if not eligible:

        if not available:

            return None, {
                "required_quality":
                    required_quality,

                "eligible_models": [],

                "scores": {},

                "fallback": True
            }


        selected = max(

            available,

            key=lambda name:
            available[
                name
            ]["quality"]
        )


        return selected, {

            "required_quality":
                required_quality,

            "eligible_models": [],

            "scores": {},

            "fallback": True
        }


    # --------------------------------------------------------
    # Cost + latency optimization
    # --------------------------------------------------------

    costs = [
        p["cost"]
        for p in eligible.values()
    ]

    latencies = [
        p["latency_ms"]
        for p in eligible.values()
    ]


    min_cost = min(costs)
    max_cost = max(costs)

    min_latency = min(latencies)
    max_latency = max(latencies)


    scores = {}


    for model_name, profile in (
        eligible.items()
    ):

        if max_cost == min_cost:

            normalized_cost = 0.0

        else:

            normalized_cost = (

                profile["cost"]
                -
                min_cost

            ) / (

                max_cost
                -
                min_cost
            )


        if max_latency == min_latency:

            normalized_latency = 0.0

        else:

            normalized_latency = (

                profile["latency_ms"]
                -
                min_latency

            ) / (

                max_latency
                -
                min_latency
            )


        scores[
            model_name
        ] = (

            COST_WEIGHT
            *
            normalized_cost

            +

            LATENCY_WEIGHT
            *
            normalized_latency
        )


    selected = min(
        scores,
        key=scores.get
    )


    return selected, {

        "required_quality":
            required_quality,

        "eligible_models":
            list(
                eligible.keys()
            ),

        "scores":
            scores,

        "fallback":
            False
    }


# ============================================================
# 13. GENERATE V4 POLICY
# ============================================================

policy_rows = []


for complexity in [
    "simple",
    "medium",
    "complex"
]:

    selected, info = route_v4(
        complexity
    )


    policy_rows.append({

        "complexity":
            complexity,

        "required_quality":
            info[
                "required_quality"
            ],

        "eligible_models":
            ", ".join(
                info[
                    "eligible_models"
                ]
            ),

        "selected_model":
            selected,

        "fallback":
            info[
                "fallback"
            ]
    })


v4_policy_df = pd.DataFrame(
    policy_rows
)


policy_path = os.path.join(
    V4_DIR,
    "v4_routing_policy.csv"
)


v4_policy_df.to_csv(
    policy_path,
    index=False
)


# ============================================================
# 14. V4 METRICS
# ============================================================

average_quality = (
    feedback_df["quality"].mean()
)

average_latency = (
    feedback_df[
        "latency_ms"
    ].mean()
)

average_cost = (
    feedback_df["cost"].mean()
)

success_rate = (
    feedback_df["success"].mean()
)


summary_df = pd.DataFrame({

    "metric": [

        "average_quality",

        "average_latency_ms",

        "average_cost",

        "success_rate",

        "total_queries"
    ],

    "value": [

        average_quality,

        average_latency,

        average_cost,

        success_rate,

        len(feedback_df)
    ]
})


summary_path = os.path.join(
    V4_DIR,
    "v4_summary.csv"
)


summary_df.to_csv(
    summary_path,
    index=False
)


# ============================================================
# 15. MODEL USAGE
# ============================================================

model_usage = (

    feedback_df[
        "selected_model"
    ]

    .value_counts()

    .to_frame(
        "requests"
    )
)


model_usage[
    "percentage"
] = (

    model_usage[
        "requests"
    ]

    /

    len(feedback_df)

    *

    100
).round(2)


# ============================================================
# 16. PRINT RESULTS
# ============================================================

print("\n")
print("=" * 100)
print("V4 OBSERVED MODEL PERFORMANCE")
print("=" * 100)

print(
    performance_df.to_string(
        index=False
    )
)


print("\n")
print("=" * 100)
print("V4 FEEDBACK-DRIVEN ROUTING POLICY")
print("=" * 100)

print(
    v4_policy_df.to_string(
        index=False
    )
)


print("\n")
print("=" * 100)
print("V4 OVERALL METRICS")
print("=" * 100)

print(
    f"Average quality : {average_quality:.4f}"
)

print(
    f"Average latency : {average_latency:.2f} ms"
)

print(
    f"Average cost    : {average_cost:.4f}"
)

print(
    f"Success rate    : {success_rate:.4f}"
)


print("\n")
print("=" * 100)
print("V4 MODEL USAGE")
print("=" * 100)

print(
    model_usage
)


# ============================================================
# 17. OPTIONAL S3 UPLOAD
# ============================================================
#
# Put your bucket name below.
#
# Leave it as None if you do not want to upload yet.
# ============================================================

S3_BUCKET = None

# Example:
# S3_BUCKET = "my-adaptive-llm-router-bucket"


if S3_BUCKET is not None:

    print("\n")
    print("=" * 100)
    print("UPLOADING V4 ARTIFACTS TO S3")
    print("=" * 100)


    s3 = boto3.client(
        "s3"
    )


    files_to_upload = [

        "routing_feedback.csv",

        "model_performance.csv",

        "model_performance.json",

        "v4_routing_policy.csv",

        "v4_summary.csv"
    ]


    for filename in (
        files_to_upload
    ):

        local_path = os.path.join(
            V4_DIR,
            filename
        )


        s3_key = (
            "adaptive-llm-router/"
            "v4/"
            +
            filename
        )


        s3.upload_file(
            local_path,
            S3_BUCKET,
            s3_key
        )


        print(
            f"Uploaded: "
            f"s3://{S3_BUCKET}/{s3_key}"
        )


# ============================================================
# 18. FINAL ARTIFACTS
# ============================================================

print("\n")
print("=" * 100)
print("V4 COMPLETE")
print("=" * 100)

print(
    "\nV4 files:"
)

for filename in [

    "routing_feedback.csv",

    "model_performance.csv",

    "model_performance.json",

    "v4_routing_policy.csv",

    "v4_summary.csv"

]:

    print(
        os.path.join(
            V4_DIR,
            filename
        )
    )


print("\nV4 is now using observed feedback to build")
print("the model-performance profile instead of relying")
print("directly on the static V3 profiles.")

V4 — FEEDBACK-DRIVEN LLM ROUTER

Evaluation dataset:
/home/sagemaker-user/adaptive-llm-routing/notebooks/data/evaluation/v1_complexity_labels.json

Total evaluation queries: 639

Complexity distribution:
complexity
simple     213
medium     213
complex    213
Name: count, dtype: int64


V4 OBSERVED MODEL PERFORMANCE
complexity selected_model  queries  avg_quality  avg_latency_ms  avg_cost  success_rate
   complex    large_model      213       0.9584         1215.70    1.0235        1.0000
    medium   medium_model      213       0.8986          506.12    0.3091        0.9906
    simple    small_model      213       0.9051          188.60    0.0903        0.9906


V4 FEEDBACK-DRIVEN ROUTING POLICY
complexity  required_quality eligible_models selected_model  fallback
    simple              0.80     small_model    small_model     False
    medium              0.90                   medium_model      True
   complex              0.95     large_model    large_model     False


V4 OVERALL M

In [2]:
# ============================================================
# V4 — FINAL FEEDBACK-DRIVEN LLM ROUTER
# ============================================================
#
# V4:
#   639 queries
#       ↓
#   Evaluate ALL 3 models
#       ↓
#   1,917 feedback records
#       ↓
#   70% query-level training
#       ↓
#   Learn model × complexity performance
#       ↓
#   Build V4 routing policy
#       ↓
#   30% query-level holdout
#       ↓
#   Compare V3 vs V4
#       ↓
#   Save artifacts
#       ↓
#   Upload artifacts to Amazon S3
#
# AWS used:
#   SageMaker environment + Amazon S3
#
# IMPORTANT:
#   Model outcomes are simulated in this version.
#   Later, the simulation can be replaced with real inference.
# ============================================================

import os
import json
import boto3
import numpy as np
import pandas as pd

from botocore.exceptions import ClientError


# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT_ROOT = (
    "/home/sagemaker-user/"
    "adaptive-llm-routing"
)

EVAL_PATH = os.path.join(
    PROJECT_ROOT,
    "notebooks",
    "data",
    "evaluation",
    "v1_complexity_labels.json"
)

V4_DIR = os.path.join(
    PROJECT_ROOT,
    "notebooks",
    "data",
    "evaluation",
    "v4"
)

os.makedirs(
    V4_DIR,
    exist_ok=True
)


RANDOM_SEED = 42

TRAIN_FRACTION = 0.70

COST_WEIGHT = 0.55

LATENCY_WEIGHT = 0.45

SUCCESS_THRESHOLD = 0.95


print("=" * 100)
print("V4 — FINAL FEEDBACK-DRIVEN LLM ROUTER")
print("=" * 100)


# ============================================================
# 2. AWS SESSION
# ============================================================

session = boto3.Session()

region = session.region_name

if region is None:

    raise RuntimeError(
        "Could not determine AWS region."
    )


sts = session.client(
    "sts"
)

account_id = (
    sts.get_caller_identity()
    ["Account"]
)


# ============================================================
# 3. CREATE / FIND S3 BUCKET
# ============================================================

bucket_name = (
    f"adaptive-llm-router-"
    f"{account_id}-"
    f"{region}"
).lower()


s3 = session.client(
    "s3",
    region_name=region
)


print("\nAWS region:")
print(region)

print("\nAWS account:")
print(account_id)

print("\nS3 bucket:")
print(bucket_name)


try:

    s3.head_bucket(
        Bucket=bucket_name
    )

    print(
        "\nS3 bucket already exists."
    )


except ClientError as e:

    error_code = (
        e.response
        .get("Error", {})
        .get("Code", "")
    )


    if error_code in {
        "404",
        "NoSuchBucket"
    }:

        print(
            "\nS3 bucket does not exist."
        )

        print(
            "Creating bucket..."
        )


        if region == "us-east-1":

            s3.create_bucket(
                Bucket=bucket_name
            )

        else:

            s3.create_bucket(

                Bucket=bucket_name,

                CreateBucketConfiguration={
                    "LocationConstraint":
                        region
                }
            )


        print(
            "S3 bucket created successfully."
        )


    elif error_code in {
        "403",
        "AccessDenied"
    }:

        raise PermissionError(
            "\nAccess to the S3 bucket was denied. "
            "Check the SageMaker execution role."
        ) from e

    else:

        raise


S3_PREFIX = (
    "adaptive-llm-router/v4"
)


print(
    "\nS3 location:"
)

print(
    f"s3://{bucket_name}/"
)


# ============================================================
# 4. LOAD EVALUATION DATASET
# ============================================================

if not os.path.exists(
    EVAL_PATH
):

    raise FileNotFoundError(
        f"Evaluation file not found:\n"
        f"{EVAL_PATH}"
    )


with open(
    EVAL_PATH,
    "r"
) as f:

    raw = json.load(
        f
    )


# ============================================================
# 5. PARSE DATASET
# ============================================================

if isinstance(
    raw,
    list
):

    evaluation_df = pd.DataFrame(
        raw
    )


elif isinstance(
    raw,
    dict
):

    records = []

    for key, value in raw.items():

        records.append({

            "query":
                key,

            "complexity":
                value
        })


    evaluation_df = pd.DataFrame(
        records
    )


else:

    raise ValueError(
        "Unsupported JSON format."
    )


# ============================================================
# 6. NORMALIZE DATASET
# ============================================================

possible_query_columns = [
    "query",
    "question",
    "prompt",
    "instruction",
    "text"
]

possible_label_columns = [
    "complexity",
    "label",
    "complexity_label",
    "category",
    "class"
]


query_col = None

for column in possible_query_columns:

    if column in evaluation_df.columns:

        query_col = column

        break


label_col = None

for column in possible_label_columns:

    if column in evaluation_df.columns:

        label_col = column

        break


if query_col is None:

    raise ValueError(
        "Could not identify the query column."
    )


if label_col is None:

    raise ValueError(
        "Could not identify the complexity column."
    )


evaluation_df = (
    evaluation_df.copy()
)


evaluation_df["query"] = (
    evaluation_df[
        query_col
    ]
    .astype(str)
    .str.strip()
)


def normalize_complexity(
    value
):

    value = (
        str(value)
        .lower()
        .strip()
    )


    mapping = {

        "0": "simple",
        "1": "medium",
        "2": "complex",

        "simple": "simple",
        "medium": "medium",
        "complex": "complex"
    }


    return mapping.get(
        value,
        value
    )


evaluation_df["complexity"] = (
    evaluation_df[
        label_col
    ]
    .apply(
        normalize_complexity
    )
)


valid_complexities = {
    "simple",
    "medium",
    "complex"
}


evaluation_df = evaluation_df[
    evaluation_df[
        "complexity"
    ].isin(
        valid_complexities
    )
].copy()


evaluation_df = evaluation_df[
    evaluation_df[
        "query"
    ].str.len()
    >
    0
].copy()


evaluation_df = (
    evaluation_df[
        [
            "query",
            "complexity"
        ]
    ]
    .reset_index(
        drop=True
    )
)


evaluation_df["query_id"] = (
    np.arange(
        len(
            evaluation_df
        )
    )
)


print("\n")
print("=" * 100)
print("EVALUATION DATASET")
print("=" * 100)

print(
    "Path:",
    EVAL_PATH
)

print(
    "Total queries:",
    len(
        evaluation_df
    )
)

print(
    "\nComplexity distribution:"
)

print(
    evaluation_df[
        "complexity"
    ].value_counts()
)


# ============================================================
# 7. MODEL DEFINITIONS
# ============================================================

models = [
    "small_model",
    "medium_model",
    "large_model"
]


# ============================================================
# 8. V3 STATIC MODEL PROFILES
# ============================================================

v3_model_profiles = {

    "small_model": {

        "quality":
            0.82,

        "latency_ms":
            200,

        "cost":
            0.10
    },

    "medium_model": {

        "quality":
            0.91,

        "latency_ms":
            500,

        "cost":
            0.30
    },

    "large_model": {

        "quality":
            0.97,

        "latency_ms":
            1200,

        "cost":
            1.00
    }
}


# ============================================================
# 9. QUALITY REQUIREMENTS
# ============================================================

quality_requirement = {

    "simple":
        0.80,

    "medium":
        0.90,

    "complex":
        0.95
}


# ============================================================
# 10. V3 ROUTER
# ============================================================

def v3_route(
    complexity
):

    required_quality = (
        quality_requirement[
            complexity
        ]
    )


    eligible = {}


    for model_name in models:

        profile = (
            v3_model_profiles[
                model_name
            ]
        )


        if (
            profile[
                "quality"
            ]
            >=
            required_quality
        ):

            eligible[
                model_name
            ] = profile


    # --------------------------------------------------------
    # Safety fallback
    # --------------------------------------------------------

    if not eligible:

        return max(

            models,

            key=lambda name:
            v3_model_profiles[
                name
            ]["quality"]
        )


    # --------------------------------------------------------
    # Cost + latency normalization
    # --------------------------------------------------------

    costs = [
        profile[
            "cost"
        ]
        for profile in eligible.values()
    ]


    latencies = [
        profile[
            "latency_ms"
        ]
        for profile in eligible.values()
    ]


    min_cost = min(
        costs
    )

    max_cost = max(
        costs
    )


    min_latency = min(
        latencies
    )

    max_latency = max(
        latencies
    )


    scores = {}


    for model_name, profile in (
        eligible.items()
    ):

        if (
            max_cost
            ==
            min_cost
        ):

            normalized_cost = 0.0

        else:

            normalized_cost = (
                profile[
                    "cost"
                ]
                -
                min_cost
            ) / (
                max_cost
                -
                min_cost
            )


        if (
            max_latency
            ==
            min_latency
        ):

            normalized_latency = 0.0

        else:

            normalized_latency = (
                profile[
                    "latency_ms"
                ]
                -
                min_latency
            ) / (
                max_latency
                -
                min_latency
            )


        scores[
            model_name
        ] = (

            COST_WEIGHT
            *
            normalized_cost

            +

            LATENCY_WEIGHT
            *
            normalized_latency
        )


    return min(
        scores,
        key=scores.get
    )


# ============================================================
# 11. SIMULATED TRUE MODEL PERFORMANCE
# ============================================================
#
# These represent the outcomes we would eventually obtain
# from real LLM calls.
#
# V4 LEARNS from generated feedback.
# It does not directly route using these values.
# ============================================================

simulated_performance = {

    "simple": {

        "small_model": {

            "quality":
                0.91,

            "latency_ms":
                190,

            "cost":
                0.09
        },

        "medium_model": {

            "quality":
                0.94,

            "latency_ms":
                480,

            "cost":
                0.29
        },

        "large_model": {

            "quality":
                0.98,

            "latency_ms":
                1150,

            "cost":
                0.98
        }
    },


    "medium": {

        "small_model": {

            "quality":
                0.76,

            "latency_ms":
                210,

            "cost":
                0.10
        },

        "medium_model": {

            "quality":
                0.91,

            "latency_ms":
                510,

            "cost":
                0.31
        },

        "large_model": {

            "quality":
                0.97,

            "latency_ms":
                1190,

            "cost":
                1.00
        }
    },


    "complex": {

        "small_model": {

            "quality":
                0.55,

            "latency_ms":
                220,

            "cost":
                0.11
        },

        "medium_model": {

            "quality":
                0.80,

            "latency_ms":
                530,

            "cost":
                0.32
        },

        "large_model": {

            "quality":
                0.96,

            "latency_ms":
                1220,

            "cost":
                1.03
        }
    }
}


# ============================================================
# 12. GENERATE FULL FEEDBACK MATRIX
# ============================================================
#
# 639 queries × 3 models = 1,917 observations
# ============================================================

rng = np.random.default_rng(
    RANDOM_SEED
)


feedback = []


for _, row in (
    evaluation_df.iterrows()
):

    query_id = int(
        row[
            "query_id"
        ]
    )

    query = row[
        "query"
    ]

    complexity = row[
        "complexity"
    ]


    for model_name in models:

        base = (
            simulated_performance[
                complexity
            ][
                model_name
            ]
        )


        # ----------------------------------------------------
        # Simulated quality
        # ----------------------------------------------------

        quality = rng.normal(

            loc=base[
                "quality"
            ],

            scale=0.04
        )


        quality = float(
            np.clip(
                quality,
                0.0,
                1.0
            )
        )


        # ----------------------------------------------------
        # Simulated latency
        # ----------------------------------------------------

        latency = rng.normal(

            loc=base[
                "latency_ms"
            ],

            scale=(
                base[
                    "latency_ms"
                ]
                *
                0.08
            )
        )


        latency = float(
            max(
                1.0,
                latency
            )
        )


        # ----------------------------------------------------
        # Simulated cost
        # ----------------------------------------------------

        cost = rng.normal(

            loc=base[
                "cost"
            ],

            scale=(
                base[
                    "cost"
                ]
                *
                0.05
            )
        )


        cost = float(
            max(
                0.001,
                cost
            )
        )


        # ----------------------------------------------------
        # Simulated success
        # ----------------------------------------------------

        success_probability = (
            0.95
            +
            0.05 * quality
        )


        success = int(
            rng.random()
            <
            success_probability
        )


        feedback.append({

            "query_id":
                query_id,

            "query":
                query,

            "complexity":
                complexity,

            "model":
                model_name,

            "quality":
                round(
                    quality,
                    4
                ),

            "latency_ms":
                round(
                    latency,
                    2
                ),

            "cost":
                round(
                    cost,
                    4
                ),

            "success":
                success
        })


feedback_df = pd.DataFrame(
    feedback
)


# ============================================================
# 13. VERIFY FEEDBACK MATRIX
# ============================================================

expected_records = (
    len(
        evaluation_df
    )
    *
    len(
        models
    )
)


if (
    len(
        feedback_df
    )
    !=
    expected_records
):

    raise RuntimeError(
        f"Expected "
        f"{expected_records} "
        f"feedback records, but got "
        f"{len(feedback_df)}."
    )


print("\n")
print("=" * 100)
print("FEEDBACK MATRIX")
print("=" * 100)

print(
    "Expected records:",
    expected_records
)

print(
    "Generated records:",
    len(
        feedback_df
    )
)

print(
    "\nRecords per complexity/model:"
)

print(
    feedback_df
    .groupby(
        [
            "complexity",
            "model"
        ]
    )
    .size()
    .unstack()
)


# ============================================================
# 14. QUERY-LEVEL 70/30 TRAIN-HOLDOUT SPLIT
# ============================================================
#
# IMPORTANT FIX:
# .to_numpy(copy=True) guarantees a writable array.
# ============================================================

rng_split = np.random.default_rng(
    RANDOM_SEED
)


train_query_ids = []

holdout_query_ids = []


for complexity in [
    "simple",
    "medium",
    "complex"
]:

    ids = (
        evaluation_df.loc[
            evaluation_df[
                "complexity"
            ]
            ==
            complexity,

            "query_id"
        ]
        .to_numpy(
            copy=True
        )
    )


    rng_split.shuffle(
        ids
    )


    split_index = int(
        len(ids)
        *
        TRAIN_FRACTION
    )


    train_query_ids.extend(
        ids[
            :split_index
        ]
    )


    holdout_query_ids.extend(
        ids[
            split_index:
        ]
    )


train_query_ids = set(
    train_query_ids
)

holdout_query_ids = set(
    holdout_query_ids
)


train_feedback = (
    feedback_df[
        feedback_df[
            "query_id"
        ].isin(
            train_query_ids
        )
    ]
    .copy()
)


holdout_feedback = (
    feedback_df[
        feedback_df[
            "query_id"
        ].isin(
            holdout_query_ids
        )
    ]
    .copy()
)


print("\n")
print("=" * 100)
print("TRAIN / HOLDOUT SPLIT")
print("=" * 100)

print(
    "Training queries:",
    len(
        train_query_ids
    )
)

print(
    "Holdout queries:",
    len(
        holdout_query_ids
    )
)

print(
    "Training feedback records:",
    len(
        train_feedback
    )
)

print(
    "Holdout feedback records:",
    len(
        holdout_feedback
    )
)


# ============================================================
# 15. LEARN OBSERVED MODEL PERFORMANCE
# ============================================================

performance_df = (

    train_feedback

    .groupby(
        [
            "complexity",
            "model"
        ]
    )

    .agg(

        queries=(
            "query_id",
            "nunique"
        ),

        avg_quality=(
            "quality",
            "mean"
        ),

        avg_latency_ms=(
            "latency_ms",
            "mean"
        ),

        avg_cost=(
            "cost",
            "mean"
        ),

        success_rate=(
            "success",
            "mean"
        )
    )

    .reset_index()
)


performance_df[
    "avg_quality"
] = (
    performance_df[
        "avg_quality"
    ]
    .round(4)
)


performance_df[
    "avg_latency_ms"
] = (
    performance_df[
        "avg_latency_ms"
    ]
    .round(2)
)


performance_df[
    "avg_cost"
] = (
    performance_df[
        "avg_cost"
    ]
    .round(4)
)


performance_df[
    "success_rate"
] = (
    performance_df[
        "success_rate"
    ]
    .round(4)
)


print("\n")
print("=" * 100)
print("LEARNED V4 MODEL PERFORMANCE")
print("=" * 100)

print(
    performance_df.to_string(
        index=False
    )
)


# ============================================================
# 16. BUILD PERFORMANCE LOOKUP
# ============================================================

observed_performance = {}


for _, row in (
    performance_df.iterrows()
):

    complexity = row[
        "complexity"
    ]

    model_name = row[
        "model"
    ]


    if (
        complexity
        not in
        observed_performance
    ):

        observed_performance[
            complexity
        ] = {}


    observed_performance[
        complexity
    ][model_name] = {

        "queries":
            int(
                row[
                    "queries"
                ]
            ),

        "quality":
            float(
                row[
                    "avg_quality"
                ]
            ),

        "latency_ms":
            float(
                row[
                    "avg_latency_ms"
                ]
            ),

        "cost":
            float(
                row[
                    "avg_cost"
                ]
            ),

        "success_rate":
            float(
                row[
                    "success_rate"
                ]
            )
    }


# ============================================================
# 17. V4 FEEDBACK-DRIVEN ROUTER
# ============================================================

def route_v4(
    complexity
):

    required_quality = (
        quality_requirement[
            complexity
        ]
    )


    available = (
        observed_performance[
            complexity
        ]
    )


    # --------------------------------------------------------
    # Quality + reliability filtering
    # --------------------------------------------------------

    eligible = {}


    for model_name, profile in (
        available.items()
    ):

        if (

            profile[
                "quality"
            ]
            >=
            required_quality

            and

            profile[
                "success_rate"
            ]
            >=
            SUCCESS_THRESHOLD

        ):

            eligible[
                model_name
            ] = profile


    # --------------------------------------------------------
    # Fallback
    # --------------------------------------------------------

    if not eligible:

        selected = max(

            available,

            key=lambda name:
            available[
                name
            ]["quality"]
        )


        return selected, {

            "required_quality":
                required_quality,

            "eligible_models":
                [],

            "scores":
                {},

            "fallback":
                True
        }


    # --------------------------------------------------------
    # Cost + latency optimization
    # --------------------------------------------------------

    costs = [
        profile[
            "cost"
        ]
        for profile in eligible.values()
    ]


    latencies = [
        profile[
            "latency_ms"
        ]
        for profile in eligible.values()
    ]


    min_cost = min(
        costs
    )

    max_cost = max(
        costs
    )


    min_latency = min(
        latencies
    )

    max_latency = max(
        latencies
    )


    scores = {}


    for model_name, profile in (
        eligible.items()
    ):

        if (
            max_cost
            ==
            min_cost
        ):

            normalized_cost = 0.0

        else:

            normalized_cost = (
                profile[
                    "cost"
                ]
                -
                min_cost
            ) / (
                max_cost
                -
                min_cost
            )


        if (
            max_latency
            ==
            min_latency
        ):

            normalized_latency = 0.0

        else:

            normalized_latency = (
                profile[
                    "latency_ms"
                ]
                -
                min_latency
            ) / (
                max_latency
                -
                min_latency
            )


        scores[
            model_name
        ] = (

            COST_WEIGHT
            *
            normalized_cost

            +

            LATENCY_WEIGHT
            *
            normalized_latency
        )


    selected = min(
        scores,
        key=scores.get
    )


    return selected, {

        "required_quality":
            required_quality,

        "eligible_models":
            list(
                eligible.keys()
            ),

        "scores":
            scores,

        "fallback":
            False
    }


# ============================================================
# 18. CREATE V4 POLICY
# ============================================================

policy_rows = []


for complexity in [
    "simple",
    "medium",
    "complex"
]:

    selected, info = (
        route_v4(
            complexity
        )
    )


    policy_rows.append({

        "complexity":
            complexity,

        "required_quality":
            info[
                "required_quality"
            ],

        "eligible_models":
            ", ".join(
                info[
                    "eligible_models"
                ]
            ),

        "selected_model":
            selected,

        "fallback":
            info[
                "fallback"
            ]
    })


v4_policy_df = pd.DataFrame(
    policy_rows
)


print("\n")
print("=" * 100)
print("V4 FEEDBACK-DRIVEN POLICY")
print("=" * 100)

print(
    v4_policy_df.to_string(
        index=False
    )
)


# ============================================================
# 19. V4 HOLDOUT EVALUATION
# ============================================================

holdout_queries = (
    holdout_feedback[
        [
            "query_id",
            "query",
            "complexity"
        ]
    ]
    .drop_duplicates(
        subset=[
            "query_id"
        ]
    )
)


v4_results = []


for _, row in (
    holdout_queries.iterrows()
):

    query_id = int(
        row[
            "query_id"
        ]
    )

    query = row[
        "query"
    ]

    complexity = row[
        "complexity"
    ]


    selected_model, info = (
        route_v4(
            complexity
        )
    )


    actual = holdout_feedback[

        (
            holdout_feedback[
                "query_id"
            ]
            ==
            query_id
        )

        &

        (
            holdout_feedback[
                "model"
            ]
            ==
            selected_model
        )
    ]


    if len(actual) != 1:

        raise RuntimeError(
            "Expected exactly one "
            "holdout observation."
        )


    actual = actual.iloc[0]


    v4_results.append({

        "query_id":
            query_id,

        "query":
            query,

        "complexity":
            complexity,

        "selected_model":
            selected_model,

        "quality":
            actual[
                "quality"
            ],

        "latency_ms":
            actual[
                "latency_ms"
            ],

        "cost":
            actual[
                "cost"
            ],

        "success":
            actual[
                "success"
            ],

        "fallback":
            info[
                "fallback"
            ]
    })


v4_results_df = pd.DataFrame(
    v4_results
)


# ============================================================
# 20. V4 HOLDOUT METRICS
# ============================================================

v4_quality = (
    v4_results_df[
        "quality"
    ].mean()
)


v4_latency = (
    v4_results_df[
        "latency_ms"
    ].mean()
)


v4_cost = (
    v4_results_df[
        "cost"
    ].mean()
)


v4_success = (
    v4_results_df[
        "success"
    ].mean()
)


v4_fallback = (
    v4_results_df[
        "fallback"
    ].mean()
)


# ============================================================
# 21. V3 BASELINE ON SAME HOLDOUT
# ============================================================

v3_results = []


for _, row in (
    holdout_queries.iterrows()
):

    query_id = int(
        row[
            "query_id"
        ]
    )

    complexity = row[
        "complexity"
    ]


    selected_model = v3_route(
        complexity
    )


    actual = holdout_feedback[

        (
            holdout_feedback[
                "query_id"
            ]
            ==
            query_id
        )

        &

        (
            holdout_feedback[
                "model"
            ]
            ==
            selected_model
        )
    ]


    if len(actual) != 1:

        raise RuntimeError(
            "Expected exactly one "
            "V3 holdout observation."
        )


    actual = actual.iloc[0]


    v3_results.append({

        "query_id":
            query_id,

        "selected_model":
            selected_model,

        "quality":
            actual[
                "quality"
            ],

        "latency_ms":
            actual[
                "latency_ms"
            ],

        "cost":
            actual[
                "cost"
            ],

        "success":
            actual[
                "success"
            ]
    })


v3_results_df = pd.DataFrame(
    v3_results
)


v3_quality = (
    v3_results_df[
        "quality"
    ].mean()
)


v3_latency = (
    v3_results_df[
        "latency_ms"
    ].mean()
)


v3_cost = (
    v3_results_df[
        "cost"
    ].mean()
)


v3_success = (
    v3_results_df[
        "success"
    ].mean()
)


# ============================================================
# 22. V3 VS V4 COMPARISON
# ============================================================

comparison_df = pd.DataFrame({

    "metric": [

        "average_quality",

        "average_latency_ms",

        "average_cost",

        "success_rate"
    ],

    "V3": [

        v3_quality,

        v3_latency,

        v3_cost,

        v3_success
    ],

    "V4": [

        v4_quality,

        v4_latency,

        v4_cost,

        v4_success
    ]
})


comparison_df[
    "V3"
] = (
    comparison_df[
        "V3"
    ]
    .round(4)
)


comparison_df[
    "V4"
] = (
    comparison_df[
        "V4"
    ]
    .round(4)
)


comparison_df[
    "delta_V4_minus_V3"
] = (
    comparison_df[
        "V4"
    ]
    -
    comparison_df[
        "V3"
    ]
).round(4)


print("\n")
print("=" * 100)
print("V3 VS V4 — HOLDOUT COMPARISON")
print("=" * 100)

print(
    comparison_df.to_string(
        index=False
    )
)


# ============================================================
# 23. V4 MODEL USAGE
# ============================================================

v4_usage = (
    v4_results_df[
        "selected_model"
    ]
    .value_counts()
    .to_frame(
        "requests"
    )
)


v4_usage[
    "percentage"
] = (
    v4_usage[
        "requests"
    ]
    /
    len(
        v4_results_df
    )
    *
    100
).round(2)


print("\n")
print("=" * 100)
print("V4 HOLDOUT MODEL USAGE")
print("=" * 100)

print(
    v4_usage
)


# ============================================================
# 24. SAVE LOCAL ARTIFACTS
# ============================================================

feedback_path = os.path.join(
    V4_DIR,
    "routing_feedback.csv"
)

feedback_df.to_csv(
    feedback_path,
    index=False
)


performance_path = os.path.join(
    V4_DIR,
    "model_performance.csv"
)

performance_df.to_csv(
    performance_path,
    index=False
)


performance_json_path = os.path.join(
    V4_DIR,
    "model_performance.json"
)

with open(
    performance_json_path,
    "w"
) as f:

    json.dump(
        observed_performance,
        f,
        indent=2
    )


policy_path = os.path.join(
    V4_DIR,
    "v4_routing_policy.csv"
)

v4_policy_df.to_csv(
    policy_path,
    index=False
)


holdout_path = os.path.join(
    V4_DIR,
    "v4_holdout_results.csv"
)

v4_results_df.to_csv(
    holdout_path,
    index=False
)


comparison_path = os.path.join(
    V4_DIR,
    "v3_vs_v4_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)


summary_df = pd.DataFrame({

    "metric": [

        "evaluation_queries",

        "training_queries",

        "holdout_queries",

        "feedback_records",

        "v3_average_quality",

        "v4_average_quality",

        "v3_average_latency_ms",

        "v4_average_latency_ms",

        "v3_average_cost",

        "v4_average_cost",

        "v3_success_rate",

        "v4_success_rate",

        "v4_fallback_rate"
    ],

    "value": [

        len(
            evaluation_df
        ),

        len(
            train_query_ids
        ),

        len(
            holdout_query_ids
        ),

        len(
            feedback_df
        ),

        v3_quality,

        v4_quality,

        v3_latency,

        v4_latency,

        v3_cost,

        v4_cost,

        v3_success,

        v4_success,

        v4_fallback
    ]
})


summary_path = os.path.join(
    V4_DIR,
    "v4_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)


# ============================================================
# 25. SAVE V4 CONFIGURATION
# ============================================================

v4_config = {

    "version":
        "V4",

    "description":
        "Feedback-driven quality/cost/latency router",

    "evaluation_queries":
        len(evaluation_df),

    "training_fraction":
        TRAIN_FRACTION,

    "random_seed":
        RANDOM_SEED,

    "quality_requirements":
        quality_requirement,

    "success_threshold":
        SUCCESS_THRESHOLD,

    "cost_weight":
        COST_WEIGHT,

    "latency_weight":
        LATENCY_WEIGHT,

    "feedback_records":
        len(feedback_df),

    "s3_bucket":
        bucket_name,

    "s3_prefix":
        S3_PREFIX,

    "simulation":
        True
}


config_path = os.path.join(
    V4_DIR,
    "v4_config.json"
)


with open(
    config_path,
    "w"
) as f:

    json.dump(
        v4_config,
        f,
        indent=2
    )


# ============================================================
# 26. UPLOAD ARTIFACTS TO S3
# ============================================================

artifacts = [

    "routing_feedback.csv",

    "model_performance.csv",

    "model_performance.json",

    "v4_routing_policy.csv",

    "v4_holdout_results.csv",

    "v3_vs_v4_comparison.csv",

    "v4_summary.csv",

    "v4_config.json"
]


print("\n")
print("=" * 100)
print("UPLOADING V4 ARTIFACTS TO S3")
print("=" * 100)


for filename in artifacts:

    local_path = os.path.join(
        V4_DIR,
        filename
    )


    s3_key = (
        S3_PREFIX
        +
        "/"
        +
        filename
    )


    s3.upload_file(
        local_path,
        bucket_name,
        s3_key
    )


    print(
        f"Uploaded: "
        f"s3://{bucket_name}/"
        f"{s3_key}"
    )


# ============================================================
# 27. FINAL STATUS
# ============================================================

print("\n")
print("=" * 100)
print("V4 COMPLETE")
print("=" * 100)

print(
    "\nEvaluation queries:",
    len(
        evaluation_df
    )
)

print(
    "Feedback records:",
    len(
        feedback_df
    )
)

print(
    "Training queries:",
    len(
        train_query_ids
    )
)

print(
    "Holdout queries:",
    len(
        holdout_query_ids
    )
)


print(
    "\nV3 holdout quality:",
    round(
        v3_quality,
        4
    )
)

print(
    "V4 holdout quality:",
    round(
        v4_quality,
        4
    )
)


print(
    "\nV3 holdout latency:",
    round(
        v3_latency,
        2
    ),
    "ms"
)

print(
    "V4 holdout latency:",
    round(
        v4_latency,
        2
    ),
    "ms"
)


print(
    "\nV3 holdout cost:",
    round(
        v3_cost,
        4
    )
)

print(
    "V4 holdout cost:",
    round(
        v4_cost,
        4
    )
)


print(
    "\nV3 success rate:",
    round(
        v3_success,
        4
    )
)

print(
    "V4 success rate:",
    round(
        v4_success,
        4
    )
)


print(
    "\nV4 fallback rate:",
    round(
        v4_fallback,
        4
    )
)


print(
    "\nS3 bucket:"
)

print(
    f"s3://{bucket_name}/"
)


print(
    "\nS3 V4 prefix:"
)

print(
    f"s3://{bucket_name}/{S3_PREFIX}/"
)


print(
    "\nLocal V4 artifacts:"
)

for filename in artifacts:

    print(
        os.path.join(
            V4_DIR,
            filename
        )
    )


print(
    "\nV4 is complete."
)

V4 — FINAL FEEDBACK-DRIVEN LLM ROUTER

AWS region:
eu-north-1

AWS account:
753176172779

S3 bucket:
adaptive-llm-router-753176172779-eu-north-1

S3 bucket already exists.

S3 location:
s3://adaptive-llm-router-753176172779-eu-north-1/


EVALUATION DATASET
Path: /home/sagemaker-user/adaptive-llm-routing/notebooks/data/evaluation/v1_complexity_labels.json
Total queries: 639

Complexity distribution:
complexity
simple     213
medium     213
complex    213
Name: count, dtype: int64


FEEDBACK MATRIX
Expected records: 1917
Generated records: 1917

Records per complexity/model:
model       large_model  medium_model  small_model
complexity                                        
complex             213           213          213
medium              213           213          213
simple              213           213          213


TRAIN / HOLDOUT SPLIT
Training queries: 447
Holdout queries: 192
Training feedback records: 1341
Holdout feedback records: 576


LEARNED V4 MODEL PERFORMANCE
compl

In [1]:
import boto3

bucket = "adaptive-llm-router-753176172779-eu-north-1"

s3 = boto3.client("s3")

s3.put_object(
    Bucket=bucket,
    Key="adaptive-llm-router/v4/permission_test.txt",
    Body=b"V4 S3 test successful"
)

print("SUCCESS: SageMaker can write to S3.")

SUCCESS: SageMaker can write to S3.


In [4]:
import sys
!{sys.executable} -m pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 95.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


V4 OBSERVED MODEL PERFORMANCE
complexity selected_model  queries  avg_quality  avg_latency_ms  avg_cost  success_rate
   complex    large_model      213       0.9584         1215.70    1.0235        1.0000
    medium   medium_model      213       0.8986          506.12    0.3091        0.9906
    simple    small_model      213       0.9051          188.60    0.0903        0.9906

Saved:
/home/sagemaker-user/adaptive-llm-routing/notebooks/data/evaluation/v4/model_performance.csv
/home/sagemaker-user/adaptive-llm-routing/notebooks/data/evaluation/v4/model_performance.json
